In [50]:
import numpy as np
from ase.units import kB
import matplotlib.pyplot as plt

In [1]:
import numpy as np
from structure import read_xyz_file, write_xyz_file, read_xyz_traj, write_xyz_traj
from many_body_potential import ml_potential, on_the_fly
from copy import deepcopy
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase import units

In [3]:
def reset_mass_center(system):
    '''
    set the mass center to [0,0,0]
    '''
    coordinates = system.get_positions()
    masses = system.get_masses()[:,np.newaxis]
    mass_center = np.sum(masses*coordinates, axis=0)/np.sum(masses)
    system.set_positions(coordinates - mass_center)

def get_translational_momentum(system):
    n_atom = system.get_global_number_of_atoms()
    return np.sum(system.get_momenta(), axis=0)/n_atom

In [10]:
target_molecule = read_xyz_file('MoREST.str_target')
MaxwellBoltzmannDistribution(target_molecule, temperature_K = 400)
Stationary(target_molecule)
reset_mass_center(target_molecule)
write_xyz_file('test_target.xyz',target_molecule)

incident_molecule = read_xyz_file('MoREST.str_incident')
MaxwellBoltzmannDistribution(incident_molecule, temperature_K = 400)
reset_mass_center(incident_molecule)
write_xyz_file('test1_incident.xyz',incident_molecule)
# redirect translational movement
scalar_translational_momentum = np.linalg.norm(get_translational_momentum(incident_molecule))
print(scalar_translational_momentum)
target_point = np.random.uniform(-1,1,3)
target_point = 1 * target_point / np.linalg.norm(target_point)
incident_point = np.random.uniform(-1,1,3)
incident_point = 9 * incident_point / np.linalg.norm(incident_point)
collision_vector = target_point - incident_point
collision_momentum = collision_vector / np.linalg.norm(collision_vector) * scalar_translational_momentum
print(incident_molecule.get_momenta())
print(get_translational_momentum(incident_molecule))
print(collision_momentum)
print(np.linalg.norm(collision_momentum))
Stationary(incident_molecule)
incident_momenta = incident_molecule.get_momenta()
print(incident_momenta)
print(incident_momenta + collision_momentum)
incident_molecule.set_momenta(incident_momenta + collision_momentum)
# move the mass center of incident molecule to the incident_point
incident_molecule.set_positions(incident_molecule.get_positions() + incident_point)
print(incident_point)
write_xyz_file('test2_incident.xyz',incident_molecule)

# combine target molecule and incident molecule
current_system = target_molecule + incident_molecule
write_xyz_file('MoREST.str', current_system)

0.3611384416932842
[[-0.16901877 -0.60231171 -0.12015926]
 [-0.01878922  0.30385541  0.2187134 ]
 [ 0.00532067  0.06766541  0.16970743]
 [ 0.62525854  1.53679705  0.16189785]]
[0.1106928  0.32650154 0.10753986]
[ 0.01880411  0.15525991 -0.32551765]
0.3611384416932843
[[-0.38291607 -1.24425368 -0.32328376]
 [-0.03722942  0.28764009  0.22597828]
 [-0.01055765  0.02635286  0.17176501]
 [ 0.43070314  0.93026073 -0.07445953]]
[[-0.36411195 -1.08899377 -0.64880141]
 [-0.01842531  0.4429     -0.09953936]
 [ 0.00824646  0.18161277 -0.15375264]
 [ 0.44950725  1.08552064 -0.39997718]]
[ 0.24243956 -4.37892696  7.85914892]


T = [100]
i = 100
while i < 1000:
    i *= 1.2
    T.append(i)
T = np.array(T)

In [11]:
print(T)

[ 100.          120.          144.          172.8         207.36
  248.832       298.5984      358.31808     429.981696    515.9780352
  619.17364224  743.00837069  891.61004483 1069.93205379]


In [12]:
print(T[1:])
print(T[:-1])
delta = 1/(kB*T[1:]) - 1/(kB*T[:-1])
print(delta)
np.exp(delta)

[ 120.          144.          172.8         207.36        248.832
  298.5984      358.31808     429.981696    515.9780352   619.17364224
  743.00837069  891.61004483 1069.93205379]
[100.         120.         144.         172.8        207.36
 248.832      298.5984     358.31808    429.981696   515.9780352
 619.17364224 743.00837069 891.61004483]
[-19.3408701  -16.11739175 -13.43115979 -11.19263316  -9.3271943
  -7.77266192  -6.47721826  -5.39768189  -4.49806824  -3.7483902
  -3.1236585   -2.60304875  -2.16920729]


array([3.98443586e-09, 1.00070415e-07, 1.46866051e-06, 1.37753036e-05,
       8.89715146e-05, 4.21090865e-04, 1.53808328e-03, 4.52706303e-03,
       1.11304772e-02, 2.35556352e-02, 4.39959146e-02, 7.40474815e-02,
       1.14268162e-01])

In [21]:
np.sign(23.233545)

1.0

In [15]:
z[1:]**2

array([1.081975  , 0.04896079])

In [ ]:
#this is step A
def position_update(x,v,dt):
    x_new = x + v*dt/2.
    return x_new

#this is step B
def velocity_update(v,F,dt):
    v_new = v + F*dt/2.
    return v_new

def random_velocity_update(v,gamma,kBT,dt):
    R = np.random.normal()
    c1 = np.exp(-gamma*dt)
    c2 = np.sqrt(1-c1*c1)*np.sqrt(kBT)
    v_new = c1*v + R*c2
    return v_new


def baoab(potential, max_time, dt, gamma, kBT, initial_position, initial_velocity,
                                        save_frequency=3, **kwargs ):
    x = initial_position
    v = initial_velocity
    t = 0
    step_number = 0
    positions = []
    velocities = []
    total_energies = []
    save_times = []
    
    while(t<max_time):
        
        # B
        potential_energy, force = potential(x,**kwargs)
        v = velocity_update(v,force,dt)
        
        #A
        x = position_update(x,v,dt)

        #O
        v = random_velocity_update(v,gamma,kBT,dt)
        
        #A
        x = position_update(x,v,dt)
        
        # B
        potential_energy, force = potential(x,**kwargs)
        v = velocity_update(v,force,dt)
        
        if step_number%save_frequency == 0 and step_number>0:
            e_total = .5*v*v + potential_energy

            positions.append(x)
            velocities.append(v)
            total_energies.append(e_total)
            save_times.append(t)
        
        t = t+dt
        step_number = step_number + 1
    
    return save_times, positions, velocities, total_energies   